# Compressible 2D: LLF vs HLL Flux Comparison

This notebook compares final density fields from the same initial condition and parameters, changing only the numerical flux scheme:
- `llf` (more diffusive, robust)
- `hll` (sharper fronts)

Panels show LLF density, HLL density, and difference (`HLL - LLF`).

In [ ]:
import matplotlib.pyplot as plt
import torch

from autosim.simulations import CompressibleFluid2D as Sim

common_kwargs = {
    "return_timeseries": True,
    "log_level": "warning",
    "n": 64,
    "T": 1.2,
    "dt_save": 0.01,
    "cfl": 0.32,
    "scenario": "vortex_sheet",
    "parameters_range": {
        "gamma": (1.4, 1.4),
        "amp": (0.18, 0.18),
    },
}

sim_llf = Sim(flux_scheme="llf", **common_kwargs)
sim_hll = Sim(flux_scheme="hll", **common_kwargs)

out_llf = sim_llf.forward_samples_spatiotemporal(n_samples=1, random_seed=42)
out_hll = sim_hll.forward_samples_spatiotemporal(n_samples=1, random_seed=42)

rho_llf = out_llf["data"][0, -1, :, :, 0]
rho_hll = out_hll["data"][0, -1, :, :, 0]
drho = rho_hll - rho_llf

vmin = float(torch.minimum(rho_llf.min(), rho_hll.min()).item())
vmax = float(torch.maximum(rho_llf.max(), rho_hll.max()).item())
abs_dmax = float(drho.abs().max().item())

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

im0 = axes[0].imshow(rho_llf.cpu(), origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
axes[0].set_title("LLF: final $\\rho$")
axes[0].set_xticks([])
axes[0].set_yticks([])
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(rho_hll.cpu(), origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
axes[1].set_title("HLL: final $\\rho$")
axes[1].set_xticks([])
axes[1].set_yticks([])
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(drho.cpu(), origin="lower", cmap="RdBu_r", vmin=-abs_dmax, vmax=abs_dmax)
axes[2].set_title("HLL - LLF")
axes[2].set_xticks([])
axes[2].set_yticks([])
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

fig.suptitle("Flux-scheme comparison on identical initial conditions", y=1.02)
plt.show()

print("rho range LLF:", float(rho_llf.min()), float(rho_llf.max()))
print("rho range HLL:", float(rho_hll.min()), float(rho_hll.max()))
print("max |HLL - LLF|:", abs_dmax)

In [ ]:
from IPython.display import HTML

from autosim.utils import plot_spatiotemporal_video

anim = plot_spatiotemporal_video(
    out_llf["data"],
    batch_idx=0,
    channel_names=["rho", "u", "v", "p"],
)
HTML(anim.to_jshtml())

In [ ]:
anim = plot_spatiotemporal_video(
    out_hll["data"],
    batch_idx=0,
    channel_names=["rho", "u", "v", "p"],
)
HTML(anim.to_jshtml())